In [1]:
!pip install -q -U jax jaxlib
!pip install -q -U flax optax optuna scikit-learn


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
%%writefile dataset_utils.py
import csv
import torch
import numpy as np
from pathlib import Path
from torch.utils.data import Dataset

# 10k pseudo-bag: forces stochastic patch subsampling each epoch (DTFD strategy)
MAX_SAFE_CEILING = 10000

class SlideDatasetPT(Dataset):
    def __init__(self, labels_csv: str, pt_dir: str, split_ids: list):
        self.pt_dir = Path(pt_dir)
        with open(labels_csv) as f:
            all_rows = list(csv.DictReader(f))
        split_set = set(split_ids)
        self.samples = []
        for row in all_rows:
            pid = row["patient_id"]
            if pid not in split_set:
                continue
            raw_label = int(row["label"])
            # Hard assert: catches data corruption instead of silently poisoning G2
            assert raw_label in {0, 1, 2}, f"Corrupt label {raw_label} for patient {pid}"
            pt_path = self.pt_dir / f"{pid}.pt"
            if pt_path.exists():
                self.samples.append({"patient_id": pid, "label": raw_label, "pt_path": pt_path})

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        data = torch.load(s["pt_path"], weights_only=True)
        return data["features"], s["label"]


def static_collate_fn(batch):
    features, labels = zip(*batch)
    B = len(features)
    D = features[0].shape[1]
    padded = np.zeros((B, MAX_SAFE_CEILING, D), dtype=np.float32)
    masks  = np.zeros((B, MAX_SAFE_CEILING), dtype=bool)
    for i, f in enumerate(features):
        n = f.shape[0]
        if n > MAX_SAFE_CEILING:
            # torch.randperm is thread-safe; guarantees a fresh pseudo-bag every call
            idx = torch.randperm(n)[:MAX_SAFE_CEILING].numpy()
            padded[i]    = f.numpy()[idx]
            masks[i]     = True
        else:
            padded[i, :n] = f.numpy()
            masks[i, :n]  = True
    return padded, np.array(labels, dtype=np.int32), masks

Writing dataset_utils.py


In [3]:
"""
H-MIL Precision Strike v2 — TPU v5e-8
======================================
Changes from the previous 23-trial run:

  ARCHITECTURE
  - Pre-LayerNorm + residual connection added to both local and global
    transformer blocks. Eliminates training instability (epoch F1 swings).

  OPTIMIZER
  - Warmup (3 epochs) + cosine decay schedule on all 4 LR tiers.
  - Gradient clipping (global norm = 1.0) chained before multi_transform.
  - No more flat AdamW; late-epoch LR is 1% of peak.

  LOSS
  - Pure alpha-weighted cross-entropy. PolyLoss removed: top trials all had
    negative epsilon (= standard CE), so just use CE directly.

  HPO
  - 5 search parameters (was 8). Ranges narrowed to the region that produced
    the top-5 F1 scores across 23 trials.
  - hidden_dim hardcoded to 256 (appeared in 4 of 5 top trials).
  - Study seeded with Trial 18 and Trial 13 configs so we don't waste 2
    trials rediscovering known-good points.
  - N_TRIALS = 12 (2 seeded + 10 TPE). Fits in ~4 hours on TPU v5e-8.

  REMOVED
  - MixUp (biologically unsound for WSI feature space)
  - SWA (broken: fixed denominator + best epochs appear before SWA window)
  - poly_epsilon from search space

  FIXED
  - Val split: fixed stratified 80/20 hold-out (not StratifiedGroupKFold
    fold-0 which made all 30 trials tune to the same val set).
  - flax.jax_utils.replicate replaced with jax.device_put_replicated
    (compatible with Flax 0.8+).
"""

import os
import csv
import json
import collections
import multiprocessing as mp
from pathlib import Path
import numpy as np

import jax
import jax.numpy as jnp
from jax import random
import flax
import flax.linen as nn
from flax.training import train_state
from flax.training.common_utils import shard
from flax import serialization
import optax
import optuna
from sklearn.metrics import f1_score
from sklearn.model_selection import train_test_split

import torch
from torch.utils.data import DataLoader
from dataset_utils import SlideDatasetPT, static_collate_fn, MAX_SAFE_CEILING

print(f"JAX devices: {jax.devices()}")
print(f"Device count: {jax.device_count()}")

# =============================================================================
# CONFIG
# =============================================================================
KAGGLE_DATASET_DIR = "/kaggle/input/datasets/apexblue/tcga-hnsc-pt-embeddings"
LABELS_CSV         = "/kaggle/input/datasets/apexblue/tumour-grading-model-tcga-manifest/tcga_hnsc_labels.csv"
OUTPUT_DIR         = "/kaggle/working/optuna_results"

EMBED_DIM  = 1024
N_CLASSES  = 3
BATCH_SIZE = 8
SEED       = 42
N_TRIALS   = 12    # 2 seeded + 10 TPE
EPOCHS     = 40
PATIENCE   = 10
HIDDEN_DIM = 256   # Hardcoded: 4/5 top trials used this

Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)


# =============================================================================
# DATA SPLIT  (fixed stratified hold-out, NOT fold-0 of SGKF)
# =============================================================================
def get_splits(labels_csv, seed=42):
    with open(labels_csv) as f:
        rows = list(csv.DictReader(f))
    # Deduplicate by patient_id (first occurrence wins)
    seen = {}
    for r in rows:
        pid = r["patient_id"]
        if pid not in seen:
            seen[pid] = int(r["label"])
    pids   = list(seen.keys())
    labels = list(seen.values())
    train_p, val_p = train_test_split(
        pids, test_size=0.20, stratify=labels, random_state=seed
    )
    return train_p, val_p


# =============================================================================
# ARCHITECTURE  (Pre-LN + residual connections added)
# =============================================================================
class HMIL_Flax(nn.Module):
    hidden_dim:   int
    n_classes:    int   = 3
    region_size:  int   = 100
    dropout_rate: float = 0.3

    @nn.compact
    def __call__(self, x, mask, deterministic: bool):
        B, N, D = x.shape

        # --- Patch projection -------------------------------------------------
        h = nn.Dense(self.hidden_dim, name='proj')(x)
        h = nn.relu(h)

        # --- Pad to multiple of region_size -----------------------------------
        pad_len = (self.region_size - (N % self.region_size)) % self.region_size
        if pad_len > 0:
            h    = jnp.pad(h,    ((0, 0), (0, pad_len), (0, 0)))
            mask = jnp.pad(mask, ((0, 0), (0, pad_len)))
        N_new       = h.shape[1]
        num_regions = N_new // self.region_size

        # --- Local transformer (Pre-LN + residual) ----------------------------
        h_regions    = h.reshape((B * num_regions, self.region_size, self.hidden_dim))
        mask_regions = mask.reshape((B * num_regions, self.region_size))

        local_cls   = self.param('local_cls', nn.initializers.normal(0.02),
                                 (1, 1, self.hidden_dim))
        local_cls_b = jnp.broadcast_to(local_cls, (B * num_regions, 1, self.hidden_dim))
        h_regions   = jnp.concatenate([local_cls_b, h_regions], axis=1)

        cls_local      = jnp.ones((B * num_regions, 1), dtype=jnp.bool_)
        local_mask_full = jnp.concatenate([cls_local, mask_regions], axis=1)
        local_attn_mask = jnp.expand_dims(local_mask_full, (1, 2))

        # Pre-LN attention block
        h_ln_l   = nn.LayerNorm(name='local_ln')(h_regions)
        h_attn_l = nn.SelfAttention(
            num_heads=4, qkv_features=self.hidden_dim,
            dropout_rate=self.dropout_rate, name='local_trans'
        )(h_ln_l, mask=local_attn_mask, deterministic=deterministic)
        h_regions = h_regions + h_attn_l   # residual

        # CLS token output -> pooled region representation
        h_pooled = h_regions[:, 0, :].reshape((B, num_regions, self.hidden_dim))

        # --- Global transformer (Pre-LN + residual) ---------------------------
        global_valid    = mask_regions.any(axis=1).reshape((B, num_regions))
        global_cls      = self.param('global_cls', nn.initializers.normal(0.02),
                                     (1, 1, self.hidden_dim))
        global_cls_b    = jnp.broadcast_to(global_cls, (B, 1, self.hidden_dim))
        h_global        = jnp.concatenate([global_cls_b, h_pooled], axis=1)

        cls_global        = jnp.ones((B, 1), dtype=jnp.bool_)
        global_mask_full  = jnp.concatenate([cls_global, global_valid], axis=1)
        global_attn_mask  = jnp.expand_dims(global_mask_full, (1, 2))

        # Pre-LN attention block
        h_ln_g   = nn.LayerNorm(name='global_ln')(h_global)
        h_attn_g = nn.SelfAttention(
            num_heads=4, qkv_features=self.hidden_dim,
            dropout_rate=self.dropout_rate, name='global_trans'
        )(h_ln_g, mask=global_attn_mask, deterministic=deterministic)
        h_global = h_global + h_attn_g   # residual

        # --- Classifier head --------------------------------------------------
        z      = nn.Dense(self.hidden_dim // 2, name='classifier_1')(h_global[:, 0, :])
        z      = nn.relu(z)
        z      = nn.Dropout(self.dropout_rate)(z, deterministic=deterministic)
        logits = nn.Dense(self.n_classes, name='classifier_2')(z)
        return logits


# =============================================================================
# OPTUNA OBJECTIVE
# =============================================================================
def objective(trial):
    # Narrowed ranges from 23-trial evidence; multivariate TPE handles correlations
    base_lr         = trial.suggest_float("base_lr",         1.5e-4, 8e-4,  log=True)
    lr_decay_global = trial.suggest_float("lr_decay_global", 0.50,   0.63)
    lr_decay_local  = trial.suggest_float("lr_decay_local",  0.50,   0.82)
    weight_decay    = trial.suggest_float("weight_decay",    5e-6,   2e-4,  log=True)
    dropout_rate    = trial.suggest_float("dropout_rate",    0.15,   0.36)

    print(f"\n{'='*55}")
    print(f"Trial {trial.number:02d} | lr={base_lr:.2e} | wd={weight_decay:.2e} "
          f"| dr={dropout_rate:.2f}")
    print(f"  lr_decay: global={lr_decay_global:.3f} local={lr_decay_local:.3f}")
    print(f"{'='*55}")

    # --- Data -----------------------------------------------------------------
    train_ids, val_ids = get_splits(LABELS_CSV, seed=SEED)
    train_ds = SlideDatasetPT(LABELS_CSV, KAGGLE_DATASET_DIR, train_ids)
    val_ds   = SlideDatasetPT(LABELS_CSV, KAGGLE_DATASET_DIR, val_ids)

    # Inverse-frequency class weights
    counts = collections.Counter(s["label"] for s in train_ds.samples)
    total  = len(train_ds.samples)
    cw     = jnp.array(
        [total / (N_CLASSES * counts.get(c, 1)) for c in range(N_CLASSES)],
        dtype=jnp.float32
    )
    print(f"  Class weights: G1={cw[0]:.2f} G2={cw[1]:.2f} G3={cw[2]:.2f}")

    train_loader = DataLoader(
        train_ds, batch_size=BATCH_SIZE, shuffle=True,
        collate_fn=static_collate_fn, num_workers=0, drop_last=True
    )
    val_loader = DataLoader(
        val_ds, batch_size=BATCH_SIZE, shuffle=False,
        collate_fn=static_collate_fn, num_workers=0, drop_last=False
    )

    # --- LR schedule (warmup 3ep + cosine decay to 1% of peak) ---------------
    n_steps_per_epoch = len(train_loader)
    warmup_steps      = 3  * n_steps_per_epoch
    total_steps       = EPOCHS * n_steps_per_epoch

    def make_sched(peak_lr):
        return optax.warmup_cosine_decay_schedule(
            init_value=0.0,
            peak_value=peak_lr,
            warmup_steps=warmup_steps,
            decay_steps=total_steps,
            end_value=peak_lr * 0.01
        )

    # 4-tier LR: classifier > global > local > proj
    # proj gets a fixed additional 0.5x factor (not searched to keep dims down)
    lr_g = base_lr * lr_decay_global
    lr_l = lr_g    * lr_decay_local
    lr_p = lr_l    * 0.5

    # --- Model init -----------------------------------------------------------
    model = HMIL_Flax(hidden_dim=HIDDEN_DIM, dropout_rate=dropout_rate)
    rng   = random.PRNGKey(SEED + trial.number)
    rng, init_rng = random.split(rng)

    dummy_x    = jnp.ones((BATCH_SIZE, MAX_SAFE_CEILING, EMBED_DIM))
    dummy_mask = jnp.ones((BATCH_SIZE, MAX_SAFE_CEILING), dtype=jnp.bool_)
    params     = model.init(init_rng, dummy_x, dummy_mask, deterministic=True)['params']

    # --- Layer-wise optimizer with gradient clipping --------------------------
    flat_params = flax.traverse_util.flatten_dict(params)

    def map_layers(path):
        name = '/'.join(path)
        if 'classifier' in name: return 'classifier'
        if 'global'     in name: return 'global_layer'
        if 'local'      in name: return 'local_layer'
        return 'proj_layer'

    param_labels = flax.traverse_util.unflatten_dict(
        {k: map_layers(k) for k in flat_params}
    )

    tx = optax.chain(
        optax.clip_by_global_norm(1.0),          # clip before moment updates
        optax.multi_transform(
            {
                'classifier':  optax.adamw(make_sched(base_lr), weight_decay=weight_decay),
                'global_layer':optax.adamw(make_sched(lr_g),    weight_decay=weight_decay),
                'local_layer': optax.adamw(make_sched(lr_l),    weight_decay=weight_decay),
                'proj_layer':  optax.adamw(make_sched(lr_p),    weight_decay=weight_decay),
            },
            param_labels=param_labels
        )
    )

    state = train_state.TrainState.create(
        apply_fn=model.apply, params=params, tx=tx
    )
    # Replicate across all 8 TPU cores (replaces deprecated flax.jax_utils.replicate)
    state   = jax.device_put_replicated(state,  jax.local_devices())
    cw_pmap = jax.device_put_replicated(cw,     jax.local_devices())

    # --- Train step (pure weighted CE) ----------------------------------------
    def train_step(state, feats, masks, labels, weights, rng_drop):
        def loss_fn(params):
            logits  = state.apply_fn(
                {'params': params}, feats, masks,
                deterministic=False, rngs={'dropout': rng_drop}
            )
            one_hot = jax.nn.one_hot(labels, N_CLASSES)
            ce      = optax.softmax_cross_entropy(logits=logits, labels=one_hot)
            # Per-sample class weighting
            return jnp.mean(ce * weights[labels])

        grads     = jax.grad(loss_fn)(state.params)
        grads     = jax.lax.pmean(grads, axis_name='batch')
        new_state = state.apply_gradients(grads=grads)
        return new_state

    train_step = jax.pmap(train_step, axis_name='batch')

    # --- Eval step ------------------------------------------------------------
    def eval_step(params, feats, masks):
        logits = model.apply({'params': params}, feats, masks, deterministic=True)
        return jnp.argmax(logits, axis=-1)

    eval_step = jax.pmap(eval_step, axis_name='batch')

    # --- Training loop --------------------------------------------------------
    best_val_f1 = 0.0
    no_improve  = 0
    best_params = None

    for epoch in range(1, EPOCHS + 1):

        # Train
        for feats, labels, masks in train_loader:
            rng, step_rng = random.split(rng)
            # One PRNGKey per device for independent dropout per core
            step_rngs = random.split(step_rng, jax.device_count())
            state = train_step(
                state, shard(feats), shard(masks), shard(labels), cw_pmap, step_rngs
            )

        # Validate
        all_preds, all_labels = [], []
        for feats, labels, masks in val_loader:
            n   = labels.shape[0]
            rem = n % jax.device_count()
            if rem != 0:
                pad   = jax.device_count() - rem
                feats = np.pad(feats, ((0, pad), (0, 0), (0, 0)))
                masks = np.pad(masks, ((0, pad), (0, 0)))
            preds = eval_step(state.params, shard(feats), shard(masks))
            all_preds.extend(np.array(preds).flatten()[:n])
            all_labels.extend(labels)

        val_f1 = f1_score(all_labels, all_preds, average='macro')

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            no_improve  = 0
            # Save unreplicated params from device 0
            best_params = jax.tree_util.tree_map(lambda x: x[0], state.params)
        else:
            no_improve += 1

        print(f"Epoch {epoch:02d}/{EPOCHS} | F1: {val_f1:.4f} | Patience: {PATIENCE - no_improve}")
        trial.report(val_f1, epoch)

        if no_improve >= PATIENCE:
            print(f"Early stopping at Epoch {epoch}.")
            break

        if trial.should_prune():
            print(f"Trial {trial.number} pruned at Epoch {epoch}.")
            raise optuna.exceptions.TrialPruned()

    print(f"Trial {trial.number} → Best F1: {best_val_f1:.4f}")

    # --- Global champion save -------------------------------------------------
    if best_val_f1 > getattr(objective, "global_best_f1", 0.0):
        objective.global_best_f1 = best_val_f1
        print(f"\n🌟 NEW CHAMPION! F1={best_val_f1:.4f}. Saving...")
        if best_params is not None:
            with open(f"{OUTPUT_DIR}/global_best_weights.msgpack", "wb") as f:
                f.write(serialization.to_bytes(best_params))
        with open(f"{OUTPUT_DIR}/global_best_params.json", "w") as f:
            json.dump(trial.params, f, indent=2)

    return best_val_f1


objective.global_best_f1 = 0.0


# =============================================================================
# STUDY
# =============================================================================
if __name__ == "__main__":
    mp.set_start_method('spawn', force=True)

    print("\n" + "="*55)
    print("H-MIL PRECISION STRIKE v2 — 12 TRIALS")
    print("Seeded from Trial 18 + Trial 13 (prev best)")
    print("="*55)

    sampler = optuna.samplers.TPESampler(multivariate=True, seed=SEED)
    # Looser pruner: our early stopping does the heavy lifting already
    pruner  = optuna.pruners.MedianPruner(
        n_startup_trials=4, n_warmup_steps=15, interval_steps=1
    )
    study = optuna.create_study(
        direction="maximize",
        sampler=sampler,
        pruner=pruner,
        study_name="HMIL_Precision_v2"
    )

    # Warm-start: these are Trial 18 and Trial 13 from previous run.
    # TPE builds its probability model FROM these, so trial 3+ already
    # knows the good region of the search space.
    study.enqueue_trial({   # Trial 18: F1=0.6096
        "base_lr":         6.38e-4,
        "lr_decay_global": 0.551,
        "lr_decay_local":  0.703,
        "weight_decay":    1.27e-5,
        "dropout_rate":    0.240,
    })
    study.enqueue_trial({   # Trial 13: F1=0.6031
        "base_lr":         2.83e-4,
        "lr_decay_global": 0.533,
        "lr_decay_local":  0.566,
        "weight_decay":    1.44e-5,
        "dropout_rate":    0.184,
    })

    study.optimize(objective, n_trials=N_TRIALS)

    print("\n" + "="*55)
    print(f"DONE. Best Macro F1: {study.best_value:.4f}")
    print(f"Best params: {json.dumps(study.best_params, indent=2)}")

    # --- Auto-download --------------------------------------------------------
    import shutil
    from IPython.display import FileLink, display, HTML

    # Save final study summary
    summary = {
        "best_f1":    study.best_value,
        "best_params": study.best_params,
        "all_trials": [
            {"number": t.number, "value": t.value, "params": t.params}
            for t in study.trials if t.value is not None
        ]
    }
    with open(f"{OUTPUT_DIR}/study_summary.json", "w") as f:
        json.dump(summary, f, indent=2)

    print("\nZipping results...")
    shutil.make_archive("precision_v2_results", 'zip', OUTPUT_DIR)
    display(FileLink("precision_v2_results.zip"))
    display(HTML("""
    <script>
    setTimeout(function() {
        var links = document.querySelectorAll('a[href*="precision_v2_results.zip"]');
        if (links.length > 0) links[0].click();
    }, 2000);
    </script>
    """))

/usr/local/lib/python3.12/site-packages/jax/_src/cloud_tpu_init.py:93: UserWarning: Transparent hugepages are not enabled. TPU runtime startup and shutdown time should be significantly improved on TPU v5e and newer. If not already set, you may need to enable transparent hugepages in your VM image (sudo sh -c "echo always > /sys/kernel/mm/transparent_hugepage/enabled")
  warnings.warn(
E0000 00:00:1774932460.395369      12 common_lib.cc:648] Could not set metric server port: INVALID_ARGUMENT: Could not find SliceBuilder port 8471 in any of the 0 ports provided in `tpu_process_addresses`="local"
=== Source Location Trace: === 
learning/45eac/tfrc/runtime/common_lib.cc:238
/tmp/ipykernel_12/3296392965.py:390: ExperimentalWarning: Argument ``multivariate`` is an experimental feature. The interface can change in the future.
  sampler = optuna.samplers.TPESampler(multivariate=True, seed=SEED)
[I 2026-03-31 04:47:49,495] A new study created in memory with name: HMIL_Precision_v2


JAX devices: [TpuDevice(id=0, process_index=0, coords=(0,0,0), core_on_chip=0), TpuDevice(id=1, process_index=0, coords=(1,0,0), core_on_chip=0), TpuDevice(id=2, process_index=0, coords=(0,1,0), core_on_chip=0), TpuDevice(id=3, process_index=0, coords=(1,1,0), core_on_chip=0), TpuDevice(id=4, process_index=0, coords=(0,2,0), core_on_chip=0), TpuDevice(id=5, process_index=0, coords=(1,2,0), core_on_chip=0), TpuDevice(id=6, process_index=0, coords=(0,3,0), core_on_chip=0), TpuDevice(id=7, process_index=0, coords=(1,3,0), core_on_chip=0)]
Device count: 8

H-MIL PRECISION STRIKE v2 — 12 TRIALS
Seeded from Trial 18 + Trial 13 (prev best)

Trial 00 | lr=6.38e-04 | wd=1.27e-05 | dr=0.24
  lr_decay: global=0.551 local=0.703
  Class weights: G1=2.59 G2=0.55 G3=1.27


/tmp/ipykernel_12/3296392965.py:282: DeprecationWarning: jax.device_put_replicated is deprecated; use jax.device_put instead.
  state   = jax.device_put_replicated(state,  jax.local_devices())


Epoch 01/40 | F1: 0.4809 | Patience: 10
Epoch 02/40 | F1: 0.5205 | Patience: 10
Epoch 03/40 | F1: 0.5367 | Patience: 10
Epoch 04/40 | F1: 0.6163 | Patience: 10
Epoch 05/40 | F1: 0.4620 | Patience: 9
Epoch 06/40 | F1: 0.5276 | Patience: 8
Epoch 07/40 | F1: 0.5653 | Patience: 7
Epoch 08/40 | F1: 0.5409 | Patience: 6
Epoch 09/40 | F1: 0.5045 | Patience: 5
Epoch 10/40 | F1: 0.5239 | Patience: 4
Epoch 11/40 | F1: 0.5866 | Patience: 3
Epoch 12/40 | F1: 0.5046 | Patience: 2
Epoch 13/40 | F1: 0.5720 | Patience: 1


[I 2026-03-31 04:54:25,753] Trial 0 finished with value: 0.6162879577513724 and parameters: {'base_lr': 0.000638, 'lr_decay_global': 0.551, 'lr_decay_local': 0.703, 'weight_decay': 1.27e-05, 'dropout_rate': 0.24}. Best is trial 0 with value: 0.6162879577513724.


...

[I 2026-03-31 05:37:19,954] Trial 10 finished with value: 0.5574621456974399 and parameters: {'base_lr': 0.0002342782828530266, 'lr_decay_global': 0.597888385381628, 'lr_decay_local': 0.7546350655115475, 'weight_decay': 0.00017228068372810083, 'dropout_rate': 0.16670921702825178}. Best is trial 2 with value: 0.6683458969173254.


Epoch 21/40 | F1: 0.5049 | Patience: 0
Early stopping at Epoch 21.
Trial 10 → Best F1: 0.5575

Trial 11 | lr=3.41e-04 | wd=3.00e-05 | dr=0.17
  lr_decay: global=0.605 local=0.754
  Class weights: G1=2.59 G2=0.55 G3=1.27
Epoch 01/40 | F1: 0.3455 | Patience: 10
Epoch 02/40 | F1: 0.4981 | Patience: 10
Epoch 03/40 | F1: 0.4794 | Patience: 9
Epoch 04/40 | F1: 0.4669 | Patience: 8
Epoch 05/40 | F1: 0.5588 | Patience: 10
Epoch 06/40 | F1: 0.4418 | Patience: 9
Epoch 07/40 | F1: 0.6009 | Patience: 10
Epoch 08/40 | F1: 0.5621 | Patience: 9
Epoch 09/40 | F1: 0.5871 | Patience: 8
Epoch 10/40 | F1: 0.5914 | Patience: 7
Epoch 11/40 | F1: 0.5620 | Patience: 6
Epoch 12/40 | F1: 0.5639 | Patience: 5
Epoch 13/40 | F1: 0.5208 | Patience: 4
Epoch 14/40 | F1: 0.5643 | Patience: 3
Epoch 15/40 | F1: 0.6329 | Patience: 10
Epoch 16/40 | F1: 0.5594 | Patience: 9
Epoch 17/40 | F1: 0.6134 | Patience: 8
Epoch 18/40 | F1: 0.5648 | Patience: 7
Epoch 19/40 | F1: 0.5798 | Patience: 6
Epoch 20/40 | F1: 0.5930 | Patienc

[I 2026-03-31 05:43:14,620] Trial 11 finished with value: 0.6328502415458938 and parameters: {'base_lr': 0.0003409787048373282, 'lr_decay_global': 0.6051102660695817, 'lr_decay_local': 0.7540533562643076, 'weight_decay': 2.995842890039884e-05, 'dropout_rate': 0.16730662419604242}. Best is trial 2 with value: 0.6683458969173254.


Epoch 25/40 | F1: 0.5818 | Patience: 0
Early stopping at Epoch 25.
Trial 11 → Best F1: 0.6329

DONE. Best Macro F1: 0.6683
Best params: {
  "base_lr": 0.00028078987855609705,
  "lr_decay_global": 0.6235928598332892,
  "lr_decay_local": 0.7342380613796495,
  "weight_decay": 4.550475813202181e-05,
  "dropout_rate": 0.18276391449291166
}

Zipping results...


/kaggle/working/precision_v2_results.zip